In [1]:
import numpy as np
import pandas as pd
import lseg.data as ld
import matplotlib.pyplot as plt

In [4]:
# fetch data from Refinitiv
ld.open_session()

df = ld.get_history(
    universe=["AAPL.O", "MSFT.O"],
    fields=["TR.PriceClose", "TR.Volume", "TR.TOTALNUMBEROFSHARESOUTSTANDING"],
    interval="monthly",
    start="2020-01-01",
    end="2026-01-01"
)

c:\Users\zhens\Desktop\AIC-ML\.venv\Lib\site-packages\lseg\data\_tools\_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
c:\Users\zhens\Desktop\AIC-ML\.venv\Lib\site-packages\lseg\data\_tools\_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
c:\Users\zhens\Desktop\AIC-ML\.venv\Lib\site-packages\lseg\data\_tools\_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)

In [7]:
def to_long(df):
    shaped_df = df.stack(level=0)
    shaped_df = shaped_df.reset_index()
    shaped_df = shaped_df.rename(columns={"level_1": "ric", "Date": "date"})
    return shaped_df

shaped_df = to_long(df)

C:\Users\zhens\AppData\Local\Temp\ipykernel_30508\178835247.py:2:FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.


In [ ]:
def replay_members(index_ric, start, end, freq="ME"):
    # 1. All joiners and leavers between start and end
    const_changes = ld.get_data(
        universe=[index_ric],
        fields=["TR.IndexJLConstituentChangeDate", "TR.IndexJLConstituentRIC", 
                "TR.IndexJLConstituentName", "TR.IndexJLConstituentituentChange", ],
        parameters={"SDATE":f"{start}","EDATE":f"{end}", 'IC':'B'}
    )
    const_changes["Date"] = pd.to_datetime(const_changes["Date"])

    # 2. Members ON the start date: the members field, not the changes field
    start_members = ld.get_data(
        universe=[index_ric], 
        fields=["TR.IndexJLConstituentRIC"], 
        parameters={"SDATE":f"{start}"}
    )
    init_list = start_members["Constituent RIC"].tolist()

    # 3. Replay history month by month
    rows = []
    prev = pd.Timestamp(start)
    for m in pd.date_range(start, end, freq=freq):
        in_window = (const_changes["Date"] > prev) & (const_changes["Date"] <= m)
        window = const_changes[in_window]

        for ric, change in zip(window["Constituent RIC"], window["change"]):
            if change == "Leaver":
                init_list.remove(ric)
            elif change == "Joiner":
                init_list.append(ric)

        for ric in init_list:
            rows.append({"Date": m, "ric": ric})
        prev = m

    return pd.DataFrame(rows)

replay_members(".NDX", "2020-01-01", "2021-01-01")

InvalidIndexError: (Series([], Name: Change, dtype: boolean), 'Constituent RIC')